In [1]:
from google.colab import files
uploaded = files.upload()

Saving 1k_stories_100_genre.csv to 1k_stories_100_genre.csv


In [15]:
import pandas as pd

In [3]:
df = pd.read_csv("1k_stories_100_genre.csv")
df.head()

,id,title,story,genre
0,457580,The Chronicles of the Cosmic Rift,"In the year 2250, Earth had made significant s...",Science Fiction
1,297904,Eldoria's Enchanted Whispers,"In a land far away, where the sun shone bright...",Fantasy
2,620436,Echoes of Whispered Shadows,"Once upon a time, in a small, tranquil town ca...",Mystery
3,634687,Emerald Amulet Chronicles Revealed,"Once upon a time in the 16th century, a small ...",Historical Adventure
4,513427,The Shadows of St. Augustine,In the sun-drenched coastal city of St. August...,Thriller


From the full dataset (`1k_stories_100_genre.csv`), the 15 genres with the highest number of stories are selected. These genres are used to construct the few-shot prompts, where two real examples from the dataset are provided for each genre. The examples help the model understand the genre's tone and writing style before generating a description of an autumn morning.

In [4]:
GENRE_COL = "genre"
TEXT_COL = "story"
TITLE_COL = "title"

In [5]:
genre_counts = df[GENRE_COL].value_counts()

In [6]:
TOP_GENRES = genre_counts.head(15).index.tolist()

In [7]:
print("Top 15 genres:")
for i, g in enumerate(TOP_GENRES, 1):
    print(f"{i:2}. {g} ({genre_counts[g]} stories)")

Top 15 genres:
 1. Historical Adventure (20 stories)
 2. Fantasy (10 stories)
 3. Science Fiction (10 stories)
 4. Mystery (10 stories)
 5. Thriller (10 stories)
 6. Historical Fiction (10 stories)
 7. Adventure (10 stories)
 8. Horror (10 stories)
 9. Comedy (10 stories)
10. Crime (10 stories)
11. Dystopian (10 stories)
12. Cyberpunk (10 stories)
13. Steampunk (10 stories)
14. Post-Apocalyptic (10 stories)
15. Fairy Tale (10 stories)


In [8]:
N_SHOTS = 2
MAX_EXAMPLE_CHARS = 350

few_shot_pool = {}

for genre in TOP_GENRES:
    subset = (
        df[df[GENRE_COL] == genre]
        .dropna(subset=[TEXT_COL])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    few_shot_pool[genre] = []
    for _, row in subset.head(N_SHOTS).iterrows():
        few_shot_pool[genre].append({
            "title": str(row.get(TITLE_COL, "")),
            "text": str(row[TEXT_COL])[:MAX_EXAMPLE_CHARS].replace("\n", " ")
        })

In [9]:
few_shot_pool[TOP_GENRES[0]]

[{'title': 'Emerald Amulet Chronicles Revealed',
  'text': 'Once upon a time in the 16th century, a small village nestled in the heart of the English countryside, far from the maddening crowd. The villagers, led by the wise and benevolent Mayor Thomas, lived in harmony, and their days were filled with laughter and joy. However, their peaceful existence was about to be shattered by a series of mysterious eve'},
 {'title': 'The Chronicles of the Golden Scroll',
  'text': 'In the heart of the ancient city of Alexandria, a mysterious artifact was unearthed by a team of archaeologists. This artifact, known as the Golden Scroll, was said to hold the secrets of an ancient civilization that was on the brink of discovering a powerful energy source. The artifact was a golden scroll, inlaid with precious gems and adorned wit'}]

### Prompt Variations

All five prompts describe the same scenario, an autumn morning written in a melancholic style, but differ in how the instruction is phrased. P1 is the most direct and minimal version, providing only the essential request. P2 conveys the same meaning through paraphrased wording, allowing us to examine whether small lexical changes affect the generated output. P3 introduces a specific literary tone by emphasizing sadness and nostalgia. P4 focuses more strongly on the emotional aspect and encourages a poetic response. Finally, P5 provides the most detailed guidance by explicitly requesting literary devices such as metaphors and sensory imagery, which is expected to produce richer and more descriptive text.

In [10]:
BASE_PROMPTS = [
    "Write a description of an autumn morning in a melancholic style.",
    "Create a short text portraying an autumn morning with a melancholic atmosphere.",
    "Describe an autumn morning using a sad nostalgic literary tone.",
    "Write a poetic paragraph about an autumn morning filled with melancholy.",
    "Write a literary description of an autumn morning using metaphors and sensory imagery.",
]

In [11]:
def build_few_shot_prompt(genre: str, base_prompt: str, n_shots: int = 2) -> str:
    examples = few_shot_pool[genre][:n_shots]

    lines = [
        f"You are writing in the genre: {genre}.",
        "Use the following real examples from the dataset only as style and genre references.",
        "Do not copy them. Write a new original text for the task.",
        "",
        "### Dataset examples"
    ]

    for i, ex in enumerate(examples, 1):
        lines.append(f"Example {i} - {genre}")
        if ex["title"]:
            lines.append(f"Title: {ex['title']}")
        lines.append(f"Text: {ex['text']}")
        lines.append("")

    lines.extend([
        "### Task",
        base_prompt,
        "Keep the same genre influence as the examples, but the topic must stay an autumn morning.",
        "Generated text:"
    ])

    return "\n".join(lines)

In [12]:
all_prompts = []
for genre in TOP_GENRES:
    for prompt_idx, base_prompt in enumerate(BASE_PROMPTS, start=1):
        all_prompts.append({
            "target_genre": genre,
            "prompt_idx": prompt_idx,
            "base_prompt": base_prompt,
            "prompt": build_few_shot_prompt(genre, base_prompt, n_shots=N_SHOTS)
        })

In [13]:
print(all_prompts[0]["prompt"])

You are writing in the genre: Historical Adventure.
Use the following real examples from the dataset only as style and genre references.
Do not copy them. Write a new original text for the task.

### Dataset examples
Example 1 - Historical Adventure
Title: Emerald Amulet Chronicles Revealed
Text: Once upon a time in the 16th century, a small village nestled in the heart of the English countryside, far from the maddening crowd. The villagers, led by the wise and benevolent Mayor Thomas, lived in harmony, and their days were filled with laughter and joy. However, their peaceful existence was about to be shattered by a series of mysterious eve

Example 2 - Historical Adventure
Title: The Chronicles of the Golden Scroll
Text: In the heart of the ancient city of Alexandria, a mysterious artifact was unearthed by a team of archaeologists. This artifact, known as the Golden Scroll, was said to hold the secrets of an ancient civilization that was on the brink of discovering a powerful energy s

In [16]:
from transformers import pipeline

In [22]:
from huggingface_hub import login
login()

In [23]:
MODEL_NAME = "gpt2"

In [24]:
generator = pipeline("text-generation", model=MODEL_NAME)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
def generate_texts(prompt_entries, samples_per_prompt=5):
    rows = []

    for entry in prompt_entries:
        for sample_id in range(1, samples_per_prompt + 1):

            raw = generator(
                entry["prompt"],
                max_new_tokens=150,
                temperature=0.85,
                do_sample=True,
                pad_token_id=generator.tokenizer.eos_token_id
            )[0]["generated_text"]

            generated_only = raw[len(entry["prompt"]):].strip()

            rows.append({
                "target_genre": entry["target_genre"],
                "prompt_idx": entry["prompt_idx"],
                "sample_id": sample_id,
                "base_prompt": entry["base_prompt"],
                "generated": generated_only
            })

    return rows

In [26]:
SAMPLES_PER_PROMPT = 5

In [27]:
results_df = pd.DataFrame(
    generate_texts(all_prompts, SAMPLES_PER_PROMPT)
)

Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

After each text is generated, a zero-shot Natural Language Inference (NLI) classifier is used to evaluate whether the output matches the intended genre. The classifier is based on the `facebook/bart-large-mnli` model and does not require additional training on the dataset. Instead, each genre is represented through a descriptive label (e.g., *"fantasy with magic, enchanted forests and mythical creatures"* or *"science fiction with space travel and futuristic technology"*). The generated text is compared against all candidate genre descriptions, and the model predicts the most likely genre by measuring semantic similarity between the text and the genre labels. This approach allows automatic verification of genre consistency and provides an objective way to assess whether the generated story aligns with the genre specified in the prompt.

In [28]:
GENRE_HINT_MAP = {
    "Historical Adventure": "historical adventure set in past centuries with quests and kingdoms",
    "Science Fiction": "science fiction with space travel, futuristic technology and alien worlds",
    "Fantasy": "fantasy with magic, enchanted forests and mythical creatures",
    "Mystery": "mystery with detectives, secrets and unsolved crimes",
    "Thriller": "thriller with danger, betrayal and high-stakes tension",
    "Historical Fiction": "historical fiction set in a specific era with period-accurate detail",
    "Adventure": "adventure with exploration, brave heroes and unknown lands",
    "Horror": "horror with ghosts, haunted places and terrifying events",
    "Comedy": "comedy with humorous characters and funny situations",
    "Crime": "crime story with criminals, investigations and urban danger",
    "Dystopian": "dystopian society with oppressive regimes and ruined civilizations",
    "Cyberpunk": "cyberpunk with neon cities, hacking and human-machine fusion",
    "Steampunk": "steampunk with Victorian-era technology and mechanical inventions",
    "Romance": "romance focused on love, relationships and emotional connection",
    "Young Adult": "young adult fiction with coming-of-age themes and teenage characters",
}

In [29]:
GENRE_HINTS = {g: GENRE_HINT_MAP.get(g, g.lower() + " story") for g in TOP_GENRES}

In [30]:
candidate_labels = list(GENRE_HINTS.values())

In [31]:
label_to_genre = {v: k for k, v in GENRE_HINTS.items()}

In [32]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [33]:
def classify_genre(text: str) -> str:
    result = classifier(
        text[:800],
        candidate_labels=candidate_labels,
        hypothesis_template="This text is a {}.",
        multi_label=False
    )
    return label_to_genre[result["labels"][0]]

In [34]:
results_df["predicted_genre"] = results_df["generated"].apply(classify_genre)
results_df["genre_match"] = results_df["target_genre"] == results_df["predicted_genre"]

In [44]:
results_df[["target_genre", "predicted_genre", "genre_match", "generated"]].tail()

,target_genre,predicted_genre,genre_match,generated
370,Fairy Tale,Fantasy,False,Create an interactive page using the template ...
371,Fairy Tale,Fairy Tale,True,// Example 1 - A Tale of the Enchanted Forest ...
372,Fairy Tale,Fairy Tale,True,Use the following formatted text to describe t...
373,Fairy Tale,Fairy Tale,True,One letter can contain all the following eleme...
374,Fairy Tale,Fantasy,False,Example 2 - The Enchanted Forest of Elaria\n\n...


In [40]:
overall = results_df["genre_match"].mean()

In [41]:
print(f"Genre Match Accuracy: {overall:.2%}")

Genre Match Accuracy: 37.07%


In [46]:
import numpy as np
import torch
import pandas as pd
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer

In [47]:
texts = results_df["generated"].dropna().astype(str).tolist()

**Cosine similarity** was used to measure the semantic similarity between the generated texts. First, each text was converted into a vector representation using the `all-MiniLM-L6-v2` embedding model. Cosine similarity was then computed for every pair of texts, and the average score was calculated. Higher similarity values indicate that the outputs convey similar meanings and themes despite differences in prompt wording.

In [48]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(texts)
sim_matrix = cosine_similarity(embeddings)

upper = sim_matrix[np.triu_indices(len(sim_matrix), k=1)]
avg_cosine = float(np.mean(upper))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Sentiment analysis** was applied to all generated texts using a pretrained sentiment classification model. For each text, the model predicted whether the sentiment was positive or negative and assigned a confidence score. To create a single numerical scale, positive predictions were recorded as positive values, while negative predictions were assigned negative values. The average sentiment score was then calculated to measure the overall emotional tone of the generated texts, while the standard deviation was used to assess the variation in sentiment across different outputs. Since all prompts requested a melancholic description of an autumn morning, the generated texts were expected to exhibit predominantly negative sentiment.

In [49]:
sent_pipe = pipeline("sentiment-analysis")

sent_scores = []

for t in texts:
    r = sent_pipe(t[:512])[0]
    score = r["score"] if r["label"] == "POSITIVE" else -r["score"]
    sent_scores.append(score)

sentiment_avg = float(np.mean(sent_scores))
sentiment_var = float(np.std(sent_scores))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

**Perplexity** was computed using a pretrained GPT-2 language model as a measure of text fluency and linguistic coherence. For each generated text, the model estimated the likelihood of the observed word sequence and converted the prediction loss into a perplexity score. Lower perplexity values correspond to more natural and predictable language, whereas higher values indicate reduced fluency or increased linguistic complexity. The variability of perplexity scores across generated texts was assessed using the standard deviation, providing insight into the robustness of the model's language generation under different prompt formulations.

In [50]:
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_mdl = GPT2LMHeadModel.from_pretrained("gpt2")

def perplexity(text):
    enc = gpt2_tok(text, return_tensors="pt")
    ids = enc.input_ids[:, :gpt2_mdl.config.n_positions]

    with torch.no_grad():
        out = gpt2_mdl(ids, labels=ids)

    return torch.exp(out.loss).item()

ppl_scores = [perplexity(t) for t in texts]
ppl_var = float(np.std(ppl_scores))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


The generated texts were evaluated using a combination of semantic, stylistic, and linguistic metrics. Cosine similarity was used to measure semantic consistency, sentiment metrics assessed the preservation of the intended melancholic tone, and perplexity variation evaluated the stability of language fluency across outputs. **Style consistency** measured how frequently the same genre was predicted among the generated texts, while genre match accuracy quantified the alignment between the intended and predicted genres.

In [51]:
style_counts = Counter(results_df["predicted_genre"].tolist())
style_consistency = style_counts.most_common(1)[0][1] / len(texts)

In [52]:
metrics = {
    "cosine_similarity": avg_cosine,
    "sentiment_avg": sentiment_avg,
    "sentiment_variation": sentiment_var,
    "perplexity_variation": ppl_var,
    "style_consistency": style_consistency,
    "genre_match_accuracy": overall,
}

df_metrics = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])

display(
    df_metrics
    .style
    .hide(axis="index")
    .format({"Value": "{:.4f}"})
)

Metric,Value
cosine_similarity,0.2610
sentiment_avg,-0.1585
sentiment_variation,0.9354
perplexity_variation,6.7028
style_consistency,0.1920
genre_match_accuracy,0.3707


The obtained results show that GPT-2 generates diverse outputs but demonstrates limited robustness in preserving the intended genre. While the generated texts exhibit stylistic variation and low similarity to one another, the genre match accuracy of 37.07% and style consistency of 19.20% indicate that the model frequently deviates from the target genre despite the use of few-shot examples.
